# YLM: options A/B/C and parity ablation

Self-contained small TensorQA pipeline on branch `feat/options-bc-oct29`. Select a GPU runtime.
By default (`FULL = True`) this trains full-size YLM Options A/B/C, a task-ID baseline, and old Option A with raw odd scalars.
The task-ID baseline removes question text; this notebook does not train the Qwen baseline.
Labels are real GFN2-xTB calculations on 20,000 QM9 molecules (same recipe as v0; labeling took 16 s with 7 workers on the dev box CPU). Set `FULL = False` for a 500-molecule / tiny-model / 400-step smoke run. All models share splits and seed.

**Runtime estimate, not a measurement on Colab:** allow roughly 15–60 minutes on a T4,
10–45 minutes on an L4, or 10–40 minutes on an A100, including downloads and CPU label/text preparation.
The repo documents ~35 minutes per full 20k-molecule/60-epoch run on V100; this notebook caps
training at 400 steps/model and uses much smaller widths. Measured two-thread CPU tiny-model
forward/backward at batch 1 is about 12–24 ms for 8–24 atoms and 4–16 tokens (see benchmark JSON).
Batch size, longer text, data preparation, shared Colab CPU, and tensor-product GPU overhead prevent
reliable linear extrapolation; these are deliberately broad planning estimates, not GPU speed claims.
Internet access to GitHub, PyPI, DeepChem, and Hugging Face is required. Drive holds outputs throughout.


**Full-mode estimate:** the repo documents ~35 min per full 20k-molecule/60-epoch run on a V100, so five runs ≈ 3 h on V100/L4-class, ~1.5–2.5 h on A100, ~5–8 h on T4 (estimates; Option C has ~13M params vs 0.25M for A, so its run is slower). Smoke mode: 15–60 min.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, time, shutil

drive.mount('/content/drive')
REPO = 'YLM'
BRANCH = 'feat/options-bc-oct29'
STAMP = time.strftime('%Y%m%d_%H%M%S')
SAVE = Path('/content/drive/MyDrive') / f'{REPO}_runs' / STAMP
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if ROOT.exists():
    raise RuntimeError('Use a fresh runtime or remove /content/YLM before rerunning setup.')
subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'e3nn==0.6.0', 'numpy<2.3',
                'scipy', 'rdkit', 'tblite', 'transformers>=4.40,<5', 'accelerate', 'pytest'], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(), 'Saving to', SAVE)
# Write results directly to Drive so interrupted runs retain completed checkpoints.
Path('runs').symlink_to(SAVE / 'runs', target_is_directory=True)
(SAVE / 'runs').mkdir(exist_ok=True)

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    subprocess.run(command, check=True)


## Build data: scripts 01–03
The download contains all QM9 geometries; only 500 are labeled. Two CPU workers run tblite. Frozen MiniLM embeddings are computed once.

In [ ]:
import os
import urllib.request, zipfile
Path('data').mkdir(exist_ok=True)
url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip'
archive = Path('data/qm9.zip')
urllib.request.urlretrieve(url, archive)
with zipfile.ZipFile(archive) as z:
    z.extractall('data')
assert Path('data/gdb9.sdf').exists()
FULL = True
N_MOLECULES = 20000 if FULL else 500
run_script('01_make_labels.py', '--n', N_MOLECULES, '--workers', os.cpu_count() or 2, '--seed', 0)
run_script('02_make_questions.py', '--seed', 0)
run_script('03_embed_text.py', '--name', 'minilm')
for name in ['qm9_xtb.pkl', 'tensorqa_v0.json', 'text_minilm.pt']:
    shutil.copy2(Path('data') / name, SAVE / name)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Train and evaluate
Script 04 saves the best validation model and evaluates held-out molecules/wordings. `_o3` results aggregate identity, rotations and their reflections; `_reflection` reports only improper transforms; compare these with rotation-only results. Old/fixed A use the same seed. Increase `MAX_STEPS` and `EPOCHS` for meaningful learning curves; these short runs cannot establish comparative model quality.

In [ ]:
P = 'full_' if FULL else 'tiny_'
MAX_STEPS = None if FULL else 400
EPOCHS = None if FULL else 20
BATCH_SIZE = None if FULL else 16
common = (['--device', 'cuda'] if FULL else ['--tiny', '--device', 'cuda', '--epochs', EPOCHS,
          '--max_steps', MAX_STEPS, '--batch_size', BATCH_SIZE]) + ['--eval_batch_size', 16, '--eval_rotations', 4,
          '--eval_reflections', '--seed', 0]
experiments = [(P+'A', 'minilm', 'A', []), (P+'B', 'minilm', 'B', []),
               (P+'C', 'minilm', 'C', []), (P+'taskid', 'taskid', 'A', []),
               (P+'A_legacy', 'minilm', 'A', ['--legacy_odd_scalars'])]
for name, text, option, extra in experiments:
    run_script('04_train.py', '--name', name, '--text', text, '--option', option, *common, *extra)
json.dump(dict(molecules=N_MOLECULES, max_steps=MAX_STEPS, epochs=EPOCHS,
               batch_size=BATCH_SIZE, branch=BRANCH, gpu=torch.cuda.get_device_name()),
          (SAVE / 'config.json').open('w'), indent=2)


In [ ]:
# Re-load each portable checkpoint and independently evaluate rotations + reflections.
for name, _, _, _ in experiments:
    command = [sys.executable, 'scripts/09_eval_checkpoint.py', '--ckpt',
               f'runs/{name}/checkpoint.pt', '--eval_rotations', '8', '--reflections', '--batch_size', '16']
    with (SAVE / f'{name}_reevaluation.log').open('w') as log:
        subprocess.run(command, check=True, stdout=log, stderr=subprocess.STDOUT)
for name, _, _, _ in experiments:
    result = json.load(open(f'runs/{name}/results.json'))
    print(name, json.dumps({key: result[key] for key in ['test', 'test_o3', 'test_reflection', 'test_heldout']}))
run_script('bench_full_model.py', '--threads', 2, '--output', SAVE / 'cpu_full_model_benchmark.json')
shutil.copy2('docs/WORKLOG_oct29b.md', SAVE / 'WORKLOG_oct29b.md')


## Download all results
The archive includes labels, questions, frozen text states, run logs, best weights, portable checkpoints and metrics. It also remains in Drive.

In [ ]:
from google.colab import files
zip_path = shutil.make_archive(str(SAVE.parent / f'YLM_results_{STAMP}'), 'zip', root_dir=SAVE)
print('Saved:', zip_path)
files.download(zip_path)
